# Fraus — BERTimbau compartilhado com três cabeças

Este notebook destila os classificadores atuais em **um encoder** com cabeças de satisfação (3), emoção (7) e ironia (2). O candidato só é exportado; nada é promovido automaticamente.

Antes de começar: em **Ambiente de execução → Alterar tipo de ambiente**, selecione uma GPU. Os três checkpoints atuais devem estar no Drive em `Meu Drive/fraus/modelos/`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Ative uma GPU no ambiente de execução do Colab.'
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Projeto e dependências

Troque `BRANCH` caso o notebook esteja em outra branch. O commit usado fica registrado junto das métricas.

In [ ]:
REPO = 'https://github.com/Joao-Tavares0312/Fraus.git'
BRANCH = 'feat/backend-multitarefa'
!rm -rf /content/Fraus
!git clone --depth 1 --branch $BRANCH $REPO /content/Fraus
%cd /content/Fraus
!pip -q install -e '.[treino,onnx]'
!git rev-parse HEAD

## 2. Drive e checkpoints professores

O notebook cria links, sem copiar 1,25 GB. Ele para se qualquer professor ou o fusor estiver ausente.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
drive.mount('/content/drive')

DRIVE_FRAUS = Path('/content/drive/MyDrive/fraus')
MODELOS_DRIVE = DRIVE_FRAUS / 'modelos'
MODELOS_LOCAL = Path('/content/Fraus/modelos')
esperados = [
    MODELOS_DRIVE / 'bertimbau-satisfacao',
    MODELOS_DRIVE / 'bertimbau-emocao',
    MODELOS_DRIVE / 'bertimbau-ironia',
    MODELOS_DRIVE / 'fusor.joblib',
]
faltando = [str(p) for p in esperados if not p.exists()]
assert not faltando, 'Arquivos ausentes no Drive:\n' + '\n'.join(faltando)
if MODELOS_LOCAL.is_symlink(): MODELOS_LOCAL.unlink()
assert not MODELOS_LOCAL.exists(), 'A pasta /content/Fraus/modelos já existe; remova-a antes de continuar.'
MODELOS_LOCAL.symlink_to(MODELOS_DRIVE, target_is_directory=True)
print('Professores:', [p.name for p in esperados])

## 3. Corpus público para destilação

Misturamos avaliações B2W, GoEmotions em português e o gerador de ironia do próprio projeto. Os rótulos não são usados na destilação; eles servem apenas para garantir diversidade textual. O corpus final é deduplicado e não contém conversas privadas.

In [ ]:
import pandas as pd

SEMENTE = 42
N_B2W = 15000
N_EMOCOES = 15000
N_IRONIA = 6000

url_b2w = 'https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv'
b2w = pd.read_csv(url_b2w)
coluna_b2w = 'review_text' if 'review_text' in b2w.columns else 'texto'
textos_b2w = b2w[coluna_b2w].dropna().astype(str).sample(min(N_B2W, len(b2w)), random_state=SEMENTE)

base_emocao = 'https://huggingface.co/datasets/antoniomenezes/go_emotions_ptbr/resolve/main'
go = pd.concat([pd.read_csv(f'{base_emocao}/goemotions_{i}_pt.csv') for i in (1, 2, 3)])
textos_go = go['texto'].dropna().astype(str).drop_duplicates().sample(min(N_EMOCOES, go['texto'].nunique()), random_state=SEMENTE)

from fraus.ingest.gerador_ironia import gerar_exemplos
textos_ironia = pd.Series([texto for texto, _ in gerar_exemplos(N_IRONIA, semente=SEMENTE)])

corpus = pd.concat([textos_b2w, textos_go, textos_ironia], ignore_index=True)
corpus = corpus.astype(str).str.strip()
corpus = corpus[corpus.str.len().between(2, 2000)].drop_duplicates()
CAMINHO_CORPUS = Path('/content/corpus_multitarefa.csv')
corpus.to_frame('texto').to_csv(CAMINHO_CORPUS, index=False)
print(f'{len(corpus):,} textos em {CAMINHO_CORPUS}')
display(corpus.sample(8, random_state=SEMENTE).to_frame('texto'))

## 4. Treino e exportação ONNX

Em uma T4, comece com lote 8. Se houver falta de memória, use lote 4; se houver bastante folga, tente 16. O candidato e as métricas são gravados diretamente no Drive.

In [ ]:
import subprocess
import sys

DESTINO = DRIVE_FRAUS / 'modelos-onnx' / 'bertimbau-multitarefa-candidato'
assert not DESTINO.exists(), f'O destino já existe: {DESTINO}. Renomeie-o para preservar o experimento anterior.'
script = Path('/content/Fraus/scripts/treinar_multitarefa.py')
assert script.is_file(), f'Script de treino ausente: {script}. A branch clonada ainda não contém a implementação multitarefa.'
subprocess.run([
    sys.executable, str(script), str(CAMINHO_CORPUS),
    '--epocas', '3', '--lote', '8', '--destino', str(DESTINO),
], cwd='/content/Fraus', check=True)
assert (DESTINO / 'model.onnx').is_file(), 'O processo terminou sem produzir model.onnx.'
print('Modelo exportado:', DESTINO / 'model.onnx')

## 5. Portão de fidelidade ponta a ponta

O comparador espera o candidato dentro do projeto. Criamos outro link para o Drive e comparamos 180 conversas determinísticas. **Só aceite zero troca de categoria.**

In [ ]:
ONNX_LOCAL = Path('/content/Fraus/modelos-onnx')
ONNX_LOCAL.mkdir(exist_ok=True)
candidato_local = ONNX_LOCAL / 'bertimbau-multitarefa-candidato'
if candidato_local.is_symlink(): candidato_local.unlink()
assert not candidato_local.exists()
candidato_local.symlink_to(DESTINO, target_is_directory=True)
!python scripts/comparar_backends.py --multitarefa --n 180

## 6. Tamanho e entrega

Este número é o tamanho do artefato, não o pico de RAM do container. A medição definitiva de RSS deve ser feita na imagem Docker antes da promoção.

In [ ]:
import shutil
assert DESTINO.is_dir(), 'O candidato não existe. Execute novamente a célula de treino e confira o primeiro erro exibido.'
grafo = DESTINO / 'model.onnx'
assert grafo.is_file(), f'Exportação incompleta: {grafo} não existe.'
tamanho_mb = sum(p.stat().st_size for p in DESTINO.rglob('*') if p.is_file()) / 1024**2
print(f'Artefato: {tamanho_mb:.1f} MB')
assert tamanho_mb < 500, 'O artefato passou de 500 MB; confira se há checkpoints extras no destino.'
arquivo_zip = shutil.make_archive('/content/bertimbau-multitarefa-candidato', 'zip', DESTINO)
print('Cópia permanente:', DESTINO)
print('ZIP temporário:', arquivo_zip)
# Para baixar também ao computador, descomente:
# from google.colab import files
# files.download(arquivo_zip)

## Critérios de promoção

Não renomeie o candidato ainda. Envie `metricas_destilacao.json`, a saída completa do comparador e o tamanho exibido acima. Depois medimos RSS/latência no container e, somente se todos os portões passarem, promovemos para `bertimbau-multitarefa`.